# TP2 — Analyse d'un championnat de football avec Pandas

**Python Avancé · Licence 3 IA · ISM École d'ingénieurs** — Enseignant : Elias BADJI

**Binôme :** Hamze Abdoulkarim ADEN · Moussa CISSÉ

---

## Partie 1 — Chargement et diagnostic

### 1.1 Chargement des fichiers

In [1]:
import pandas as pd
import numpy as np

# Chargement des quatre fichiers du dossier data/
equipes = pd.read_csv("data/equipes.csv")
matchs  = pd.read_csv("data/matchs.csv", parse_dates=["date"])
joueurs = pd.read_csv("data/joueurs.csv")
buts    = pd.read_csv("data/buts.csv")

print("equipes :", equipes.shape)
print("matchs  :", matchs.shape)
print("joueurs :", joueurs.shape)
print("buts    :", buts.shape)
print("Type de matchs['date'] :", matchs["date"].dtype)

equipes : (20, 8)
matchs  : (1145, 13)
joueurs : (1688, 15)
buts    : (2660, 8)
Type de matchs['date'] : datetime64[us]


**Interprétation :** les quatre fichiers sont chargés avec les dimensions annoncées dans l'énoncé (20 clubs, 1 145 matchs, 1 688 lignes joueur-saison, 2 660 buts) et la colonne `date` de `matchs` est bien reconnue comme une date (`datetime64`). On remarque déjà que 3 saisons × 380 matchs = 1 140 : `matchs` contient donc 5 lignes de trop, probablement des doublons.

### 1.2 Forme et types des DataFrames

In [2]:
for nom, df in [("equipes", equipes), ("matchs", matchs), ("joueurs", joueurs), ("buts", buts)]:
    print("=" * 60)
    print(nom.upper())
    df.info()

EQUIPES
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   equipe_id             20 non-null     str  
 1   nom                   20 non-null     str  
 2   ville                 20 non-null     str  
 3   region                20 non-null     str  
 4   annee_creation        20 non-null     int64
 5   stade                 20 non-null     str  
 6   capacite_stade        20 non-null     int64
 7   budget_millions_fcfa  20 non-null     int64
dtypes: int64(3), str(5)
memory usage: 1.4 KB
MATCHS
<class 'pandas.DataFrame'>
RangeIndex: 1145 entries, 0 to 1144
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   match_id            1145 non-null   str           
 1   saison              1145 non-null   str           
 2   journee             1145 non-null

**Interprétation :** `equipes` est complète et correctement typée. Dans les autres tables, plusieurs colonnes n'ont pas le bon type : `salaire_mensuel_fcfa` et `date_naissance` (joueurs) sont lues comme du texte (`str`), tout comme `minute` (buts) à cause du temps additionnel. Des valeurs manquent aussi dans `spectateurs`, `salaire_mensuel_fcfa`, `date_naissance`, `minutes_jouees` et `passeur_id`.

### 1.3 Type de la colonne `salaire_mensuel_fcfa`

In [3]:
salaire = joueurs["salaire_mensuel_fcfa"]
print("Type de la colonne :", salaire.dtype)

# Tentative de conversion : les valeurs non convertibles deviennent NaN
salaire_converti = pd.to_numeric(salaire, errors="coerce")

# Valeurs bloquantes = présentes au départ, mais NaN après conversion
bloquantes = salaire[salaire_converti.isna() & salaire.notna()]

print("Nombre de valeurs non convertibles :", len(bloquantes))
print(bloquantes.head(8))

Type de la colonne : str
Nombre de valeurs non convertibles : 40
35     450 000
70     510 000
90     450 000
124    350 000
146    430 000
325    390 000
487    210 000
506    260 000
Name: salaire_mensuel_fcfa, dtype: str


**Interprétation :** la colonne est de type texte (`str`) car 40 salaires ont été saisis avec un espace comme séparateur de milliers (ex. `"450 000"`). Python ne sait pas convertir ce format en nombre, et une seule valeur de ce genre suffit à faire lire toute la colonne comme du texte. Il faudra retirer ces espaces avant la conversion (question 2.3).

### 1.4 Valeurs manquantes

In [4]:
# Nombre de valeurs manquantes par colonne, pour chaque table
manquants = pd.concat(
    {"equipes": equipes.isna().sum(),
     "matchs": matchs.isna().sum(),
     "joueurs": joueurs.isna().sum(),
     "buts": buts.isna().sum()},
    axis=1
)
# On n'affiche que les colonnes ayant au moins un manquant dans une table
print(manquants[manquants.sum(axis=1) > 0])
print("\nTotal par table :")
print(manquants.sum())

                      equipes  matchs  joueurs   buts
spectateurs               NaN    18.0      NaN    NaN
date_naissance            NaN     NaN     10.0    NaN
salaire_mensuel_fcfa      NaN     NaN     15.0    NaN
minutes_jouees            NaN     NaN      8.0    NaN
passeur_id                NaN     NaN      NaN  935.0

Total par table :
equipes      0.0
matchs      18.0
joueurs     33.0
buts       935.0
dtype: float64


In [5]:
# Les passeurs manquants : erreur ou non ? On les croise avec le type de but
pd.crosstab(buts["type_but"], buts["passeur_id"].isna().rename("passeur_manquant"))

passeur_manquant,False,True
type_but,,
jeu,1725,652
penalty,0,283


**Interprétation :** `equipes` est complète. Il manque 18 affluences dans `matchs`, ainsi que 15 salaires, 10 dates de naissance et 8 temps de jeu dans `joueurs` : ce sont des oublis de saisie à traiter en partie 2. En revanche, les 935 `passeur_id` manquants de `buts` ne sont **pas une erreur** : l'énoncé indique que le passeur est vide quand il n'y a pas de passe décisive, et le tableau croisé le confirme puisque les 283 penalties n'ont jamais de passeur. Ces vides sont une information et seront conservés.

### 1.5 Doublons

In [6]:
# 1) Lignes entièrement dupliquées
print("Lignes en double :")
for nom, df in [("equipes", equipes), ("matchs", matchs), ("joueurs", joueurs), ("buts", buts)]:
    print(f"  {nom:8s} : {df.duplicated().sum()}")

# 2) Clés en double
print("\nClés en double :")
print("  match_id dans matchs            :", matchs.duplicated(subset=["match_id"]).sum())
print("  but_id dans buts                :", buts.duplicated(subset=["but_id"]).sum())
print("  (saison, joueur_id) dans joueurs:", joueurs.duplicated(subset=["saison", "joueur_id"]).sum())

Lignes en double :
  equipes  : 0
  matchs   : 5
  joueurs  : 8
  buts     : 6

Clés en double :
  match_id dans matchs            : 5
  but_id dans buts                : 6
  (saison, joueur_id) dans joueurs: 8


**Interprétation :** `equipes` ne contient aucun doublon. `matchs`, `joueurs` et `buts` contiennent respectivement 5, 8 et 6 lignes dupliquées, et autant de clés en double : chaque doublon est donc une copie exacte d'une ligne (saisie en double), qui pourra être supprimée sans perte d'information. Après suppression, on obtiendrait 1 140 matchs (soit 3 × 380), 1 680 lignes joueur-saison et 2 654 buts, ce qui correspond aux chiffres de contrôle de l'énoncé.

### 1.6 Valeurs distinctes de `poste`

In [7]:
# repr() entoure chaque valeur de guillemets pour rendre les espaces visibles
effectifs_postes = joueurs["poste"].map(repr).value_counts()
print(effectifs_postes)
print("\nNombre de variantes :", joueurs["poste"].nunique())

poste
'Défenseur'     536
'Milieu'        534
'Attaquant'     415
'Gardien'       179
'défenseur'       2
'Attaquant '      2
'gardien'         2
'milieu'          2
'MILIEU '         2
'Milieu  '        2
'Defenseur'       2
'DEFENSEUR'       2
'Défenseur '      2
'attaquant'       2
'ATTAQUANT '      2
' Milieu'         2
Name: count, dtype: int64

Nombre de variantes : 16


**Interprétation :** la colonne `poste` contient 16 variantes pour seulement 4 postes réels. Les 12 variantes fautives (2 lignes chacune, soit 24 lignes) viennent de trois types d'erreurs de saisie : la casse (`'MILIEU'`, `'gardien'`), des espaces parasites en début ou fin (`' Milieu'`, `'Attaquant '`) et un accent manquant (`'Defenseur'`). Sans harmonisation, un `groupby("poste")` créerait 16 groupes au lieu de 4.

### 1.7 Buts marqués dans le temps additionnel

In [8]:
temps_add = buts["minute"].str.contains(r"\+")
print("Buts dans le temps additionnel :", temps_add.sum())
print(f"Soit {temps_add.mean():.1%} des buts")
print()
print(buts.loc[temps_add, "minute"].value_counts())

Buts dans le temps additionnel : 228
Soit 8.6% des buts

minute
90+2    40
90+1    40
90+5    37
90+4    35
90+3    33
45+1    16
45+3    15
45+2    12
Name: count, dtype: int64


**Interprétation :** 228 buts (8,6 % du fichier brut) ont été marqués dans le temps additionnel, dont la grande majorité après la 90e minute. Ces minutes sont stockées en texte (« 90+3 »), ce qui empêche tout calcul : il faudra séparer la minute réglementaire et les minutes additionnelles en deux colonnes numériques (question 2.6)

### 1.8 Synthèse des problèmes détectés

- **Doublons** : 5 lignes en double dans `matchs`, 8 dans `joueurs` et 6 dans `buts`, toutes des copies exactes (→ 2.1).
- **`poste`** : 16 variantes pour 4 postes, dues à la casse, à des espaces parasites et à un accent manquant (`Defenseur`) (→ 2.2).
- **`salaire_mensuel_fcfa`** : lu comme du texte car 40 valeurs contiennent un espace séparateur de milliers (`"450 000"`) (→ 2.3).
- **Salaires manquants** : 15 valeurs absentes dans `joueurs` (→ 2.4).
- **`date_naissance`** : lue comme du texte, et 10 valeurs manquantes (→ 2.5).
- **`minute`** : stockée en texte à cause du temps additionnel (228 valeurs du type `"90+3"`), donc inutilisable pour des calculs (→ 2.6).
- **`spectateurs`** : 18 affluences manquantes dans `matchs` (→ 2.7).
- **`minutes_jouees`** : 8 valeurs manquantes dans `joueurs` ; aucune question ne les utilise, elles seront laissées telles quelles.
- **Non-problème** : les 935 `passeur_id` vides de `buts` sont normaux (but sans passe décisive, dont tous les penalties) et sont conservés.

## Partie 2 — Nettoyage

On travaille sur des copies suffixées `_c` (« clean ») : les DataFrames bruts `matchs`, `joueurs` et `buts` restent intacts pour la comparaison avant/après de la question 4.8.

### 2.1 Suppression des doublons

In [9]:
# Copies nettoyées, sans les doublons exacts
matchs_c  = matchs.drop_duplicates().copy()
joueurs_c = joueurs.drop_duplicates().copy()
buts_c    = buts.drop_duplicates().copy()

# Vérification de l'unicité des clés
assert matchs_c["match_id"].is_unique
assert buts_c["but_id"].is_unique
assert not joueurs_c.duplicated(subset=["saison", "joueur_id"]).any()

print(matchs_c.shape, joueurs_c.shape, buts_c.shape)

(1140, 13) (1680, 15) (2654, 8)


**Justification :** les doublons étant des copies exactes (autant de lignes en double que de clés en double, cf. 1.5), on peut les supprimer sans perte d'information. **Interprétation :** les trois clés sont désormais uniques et on retrouve 1 140 matchs, 1 680 lignes joueur-saison et 2 654 buts.


### 2.2 Harmonisation des postes

In [10]:
joueurs_c["poste"] = (
    joueurs_c["poste"]
    .str.strip()                              # retire les espaces au début et à la fin
    .str.capitalize()                         # 1re lettre en majuscule, le reste en minuscules
    .replace({"Defenseur": "Défenseur"})      # rétablit l'accent manquant
)
print(joueurs_c["poste"].value_counts())

poste
Défenseur    540
Milieu       540
Attaquant    420
Gardien      180
Name: count, dtype: int64


**Justification :** `strip` supprime les espaces parasites, `capitalize` uniformise la casse, et `replace` corrige l'accent manquant que les deux premières méthodes ne peuvent pas rétablir. **Interprétation :** il ne reste que 4 postes ; l'effectif est cohérent (180 gardiens, 540 défenseurs, 540 milieux, 420 attaquants, soit 9 + 27 + 27 + 21 joueurs par club et par saison).


### 2.3 Conversion des salaires

In [11]:
joueurs_c["salaire_mensuel_fcfa"] = pd.to_numeric(
    joueurs_c["salaire_mensuel_fcfa"].str.replace(" ", "")
)
print("Type :", joueurs_c["salaire_mensuel_fcfa"].dtype)
print("Manquants :", joueurs_c["salaire_mensuel_fcfa"].isna().sum())

Type : float64
Manquants : 15


**Justification :** on retire l'espace séparateur de milliers avant la conversion, sans `errors="coerce"`, afin qu'une éventuelle valeur encore invalide provoque une erreur visible au lieu d'être masquée. **Interprétation :** la colonne est désormais numérique ; il ne reste que les 15 salaires réellement manquants, traités en 2.4.

### 2.4 Imputation des salaires manquants

In [12]:
# Comparaison : la médiane des salaires varie fortement selon le poste
print(joueurs_c.groupby("poste")["salaire_mensuel_fcfa"].median())

# Médiane du même poste ET de la même saison, recopiée sur chaque ligne
mediane_groupe = joueurs_c.groupby(["saison", "poste"])["salaire_mensuel_fcfa"].transform("median")
joueurs_c["salaire_mensuel_fcfa"] = joueurs_c["salaire_mensuel_fcfa"].fillna(mediane_groupe)

print("\nSalaires manquants restants :", joueurs_c["salaire_mensuel_fcfa"].isna().sum())

poste
Attaquant    470000.0
Défenseur    355000.0
Gardien      300000.0
Milieu       390000.0
Name: salaire_mensuel_fcfa, dtype: float64

Salaires manquants restants : 0


**Justification :** la médiane est peu sensible aux salaires extrêmes. On la calcule par poste car les salaires en dépendent fortement (médiane de 300 000 FCFA pour un gardien contre 470 000 pour un attaquant), et par saison car les salaires évoluent d'une saison à l'autre : une médiane globale sous-estimerait un attaquant et surestimerait un gardien. **Interprétation :** les 15 salaires manquants sont remplacés, la colonne est complète.


### 2.5 Conversion des dates de naissance

In [13]:
joueurs_c["date_naissance"] = pd.to_datetime(joueurs_c["date_naissance"], format="%Y-%m-%d")
print("Type :", joueurs_c["date_naissance"].dtype)
print("Dates manquantes :", joueurs_c["date_naissance"].isna().sum())

Type : datetime64[us]
Dates manquantes : 10


**Justification :** on **ne remplace pas** les 10 dates manquantes : une date de naissance est une donnée d'état civil propre à chaque joueur, et inventer une valeur (médiane, date arbitraire) créerait une information fausse. Aucune analyse du TP n'utilise d'ailleurs l'âge ; les `NaT` (« Not a Time ») sont simplement ignorés par Pandas. **Interprétation :** la colonne est au format date, avec 10 valeurs manquantes conservées volontairement.


### 2.6 Découpage des minutes

In [14]:
parties = buts_c["minute"].str.split("+", expand=True)   # colonne 0 : avant le +, colonne 1 : après
buts_c["minute_num"] = parties[0].astype(int)
buts_c["temps_add"]  = parties[1].fillna(0).astype(int)  # pas de + → None → 0

print(buts_c[["minute", "minute_num", "temps_add"]].drop_duplicates("minute").sort_values(["minute_num", "temps_add"]).tail(8))
print(buts_c[["minute_num", "temps_add"]].dtypes)

    minute  minute_num  temps_add
188     88          88          0
50      89          89          0
29      90          90          0
102   90+1          90          1
33    90+2          90          2
74    90+3          90          3
64    90+4          90          4
4     90+5          90          5
minute_num    int64
temps_add     int64
dtype: object


**Justification :** séparer la minute réglementaire et le temps additionnel donne deux colonnes numériques exploitables, sans perdre l'information « but dans le temps additionnel » (temps_add > 0). **Interprétation :** « 90+3 » devient minute_num = 90 et temps_add = 3 ; un but « normal » a temps_add = 0.


### 2.7 Spectateurs manquants

In [15]:
# L'affluence dépend surtout du club qui reçoit : on compare la dispersion globale et par club
print("Écart-type global          :", round(matchs_c["spectateurs"].std()))
print("Écart-type moyen par club  :", round(matchs_c.groupby(["saison", "equipe_dom_id"])["spectateurs"].std().mean()))

mediane_club = matchs_c.groupby(["saison", "equipe_dom_id"])["spectateurs"].transform("median")
matchs_c["spectateurs"] = matchs_c["spectateurs"].fillna(mediane_club).round().astype(int)

print("Spectateurs manquants restants :", matchs_c["spectateurs"].isna().sum(), "| type :", matchs_c["spectateurs"].dtype)

Écart-type global          : 3022
Écart-type moyen par club  : 673
Spectateurs manquants restants : 0 | type : int64


**Justification :** l'affluence dépend avant tout du club qui reçoit (taille du stade, popularité) : la dispersion au sein d'un même club et d'une même saison est bien plus faible que la dispersion globale (écart-type moyen de 673 spectateurs par club, contre 3 022 sur l'ensemble des matchs). On remplace donc chaque valeur manquante par la médiane des matchs à domicile du même club la même saison, arrondie à l'entier. Supprimer ces 18 matchs n'était pas envisageable, car il faut tous les matchs pour reconstruire les classements. **Interprétation :** la colonne est complète et repasse en entiers.


### 2.8 Contrôle final

In [16]:
assert len(matchs_c)  == 1140, len(matchs_c)
assert len(joueurs_c) == 1680, len(joueurs_c)
assert len(buts_c)    == 2654, len(buts_c)
print("Contrôle OK :", len(matchs_c), "matchs,", len(joueurs_c), "lignes joueur-saison,", len(buts_c), "buts")

Contrôle OK : 1140 matchs, 1680 lignes joueur-saison, 2654 buts


**Interprétation :** les trois tables nettoyées ont exactement le nombre de lignes attendu par l'énoncé ; les données brutes (`matchs`, `joueurs`, `buts`) sont conservées intactes pour la question 4.8.

## Partie 3 — Classements des trois saisons

### 3.1 à 3.3 — Une ligne par équipe et par match

In [17]:
colonnes = ["saison", "journee", "equipe_id", "buts_pour", "buts_contre"]

# 3.1 L'équipe qui reçoit : ses buts pour sont buts_dom
dom = matchs_c.rename(columns={"equipe_dom_id": "equipe_id",
                               "buts_dom": "buts_pour",
                               "buts_ext": "buts_contre"})[colonnes]

# 3.2 L'équipe qui se déplace : ses buts pour sont buts_ext
ext = matchs_c.rename(columns={"equipe_ext_id": "equipe_id",
                               "buts_ext": "buts_pour",
                               "buts_dom": "buts_contre"})[colonnes]

# 3.3 Empilement des deux tables l'une sous l'autre
resultats = pd.concat([dom, ext], ignore_index=True)
assert len(resultats) == 2 * len(matchs_c)
print("Lignes dans resultats :", len(resultats))

Lignes dans resultats : 2280


**Interprétation :** chaque match concerne deux équipes, donc `resultats` contient 2 × 1 140 = 2 280 lignes : une par équipe et par match, vue de son côté.


### 3.4 Points, victoires, nuls, défaites

In [18]:
resultats["victoire"] = (resultats["buts_pour"] >  resultats["buts_contre"]).astype(int)
resultats["nul"]      = (resultats["buts_pour"] == resultats["buts_contre"]).astype(int)
resultats["defaite"]  = (resultats["buts_pour"] <  resultats["buts_contre"]).astype(int)
resultats["points"]   = np.select([resultats["victoire"] == 1, resultats["nul"] == 1], [3, 1], default=0)
resultats.head()

,saison,journee,equipe_id,buts_pour,buts_contre,victoire,nul,defaite,points
0,2023-2024,1,E01,1,2,0,0,1,0
1,2023-2024,1,E02,2,1,1,0,0,3
2,2023-2024,1,E03,1,1,0,1,0,1
3,2023-2024,1,E04,5,1,1,0,0,3
4,2023-2024,1,E05,3,2,1,0,0,3


**Interprétation :** chaque ligne indique désormais l'issue du match pour l'équipe concernée (une seule des trois colonnes victoire/nul/defaite vaut 1) et les points gagnés selon le barème 3-1-0.


### 3.5 et 3.6 — Agrégation, noms des clubs, tri et rang

In [19]:
# 3.5 Agrégation nommée : nom_de_colonne = (colonne_source, fonction)
classements = (
    resultats.groupby(["saison", "equipe_id"])
    .agg(J=("points", "size"),
         G=("victoire", "sum"),
         N=("nul", "sum"),
         P=("defaite", "sum"),
         BP=("buts_pour", "sum"),
         BC=("buts_contre", "sum"),
         Pts=("points", "sum"))
    .reset_index()
)
classements["Diff"] = classements["BP"] - classements["BC"]

# 3.6 Nom des clubs, tri réglementaire, rang
classements = classements.merge(equipes[["equipe_id", "nom"]], on="equipe_id", how="left")
classements = classements.sort_values(["saison", "Pts", "Diff", "BP"],
                                      ascending=[True, False, False, False]).reset_index(drop=True)
classements["rang"] = classements.groupby("saison").cumcount() + 1
classements = classements[["saison", "rang", "equipe_id", "nom", "J", "G", "N", "P", "BP", "BC", "Diff", "Pts"]]
classements.head()

,saison,rang,equipe_id,nom,J,G,N,P,BP,BC,Diff,Pts
0,2023-2024,1,E01,Étoile du Cap-Vert,38,27,7,4,71,33,38,88
1,2023-2024,2,E03,Fleuve Athletic,38,22,6,10,54,32,22,72
2,2023-2024,3,E17,Diourbel Sporting,38,21,7,10,49,39,10,70
3,2023-2024,4,E20,Banlieue Espoir,38,18,9,11,59,33,26,63
4,2023-2024,5,E04,Palmiers FC,38,17,11,10,53,46,7,62


**Interprétation :** on obtient un classement de 20 clubs par saison, trié selon le règlement (points, puis différence de buts, puis buts marqués).


### 3.7 Contrôles de cohérence

In [20]:
assert (classements["J"] == 38).all(), "Un club n'a pas joué 38 matchs"

totaux = classements.groupby("saison")[["BP", "BC", "G", "P"]].sum()
assert (totaux["BP"] == totaux["BC"]).all(), "Buts pour ≠ buts contre"
assert (totaux["G"] == totaux["P"]).all(), "Victoires ≠ défaites"

print("Tous les contrôles sont validés")
totaux

Tous les contrôles sont validés


,BP,BC,G,P
saison,,,,
2023-2024,949,949,285,285
2024-2025,843,843,294,294
2025-2026,862,862,264,264


**Interprétation :** les trois contrôles passent. Chaque club a joué 38 matchs ; et comme chaque but marqué par une équipe est encaissé par une autre, et chaque victoire correspond à une défaite de l'adversaire, les totaux sont égaux dans chaque saison. La transformation des matchs en classement n'a donc ni perdu ni dupliqué de résultat.


### 3.8 Podiums

In [21]:
podiums = classements[classements["rang"] <= 3][["saison", "rang", "nom", "Pts", "Diff"]]
print(podiums.to_string(index=False))

# Clubs présents sur le podium les trois saisons
nb_podiums = podiums.groupby("nom")["saison"].nunique()
print("\nSur le podium les 3 saisons :", list(nb_podiums[nb_podiums == 3].index))

   saison  rang                nom  Pts  Diff
2023-2024     1 Étoile du Cap-Vert   88    38
2023-2024     2    Fleuve Athletic   72    22
2023-2024     3  Diourbel Sporting   70    10
2024-2025     1        Palmiers FC   79    32
2024-2025     2 Étoile du Cap-Vert   75    37
2024-2025     3    Fleuve Athletic   71    28
2025-2026     1  Presqu'île Racing   75    32
2025-2026     2 Étoile du Cap-Vert   73    36
2025-2026     3    Baobab Sporting   67    29

Sur le podium les 3 saisons : ['Étoile du Cap-Vert']


**Interprétation :** un seul club monte sur le podium les trois saisons : l'Étoile du Cap-Vert (champion en 2023-2024, puis deuxième en 2024-2025 et 2025-2026). C'est le club le plus régulier de la Ligue, alors que le titre a changé de mains chaque saison (Étoile du Cap-Vert, Palmiers FC, Presqu'île Racing).


### 3.9 Classement 2025-2026

In [22]:
classement_2026 = classements[classements["saison"] == "2025-2026"]
display(classement_2026)

# Égalités de points ET de différence de buts
egalites = classement_2026[classement_2026.duplicated(subset=["Pts", "Diff"], keep=False)]
egalites

,saison,rang,equipe_id,nom,J,G,N,P,BP,BC,Diff,Pts
40,2025-2026,1,E09,Presqu'île Racing,38,22,9,7,58,26,32,75
41,2025-2026,2,E01,Étoile du Cap-Vert,38,21,10,7,76,40,36,73
42,2025-2026,3,E02,Baobab Sporting,38,17,16,5,60,31,29,67
43,2025-2026,4,E20,Banlieue Espoir,38,19,9,10,51,41,10,66
44,2025-2026,5,E04,Palmiers FC,38,19,7,12,54,41,13,64
45,2025-2026,6,E10,Niayes Académie,38,17,8,13,46,40,6,59
46,2025-2026,7,E13,Sine Racing,38,15,11,12,42,41,1,56
47,2025-2026,8,E05,Saloum United,38,12,17,9,44,36,8,53
48,2025-2026,9,E15,Delta Club,38,13,12,13,29,45,-16,51
49,2025-2026,10,E06,Océan Club,38,13,11,14,42,48,-6,50


,saison,rang,equipe_id,nom,J,G,N,P,BP,BC,Diff,Pts
51,2025-2026,12,E14,Ferlo United,38,12,11,15,38,43,-5,47
52,2025-2026,13,E16,Forêt FC,38,11,14,13,36,41,-5,47


**Interprétation :** Presqu'île Racing est champion 2025-2026 avec 75 points, devant l'Étoile du Cap-Vert (73), qui a pourtant la meilleure attaque et la meilleure différence de buts. Ferlo United et Forêt FC terminent à égalité parfaite (47 points, différence −5) : Ferlo United est 12e devant Forêt FC grâce au troisième critère, les buts marqués (38 contre 36).

## Partie 4 — Analyses pour le rapport

### 4.1 Meilleurs buteurs 2025-2026

In [23]:
noms_clubs = equipes[["equipe_id", "nom"]].rename(columns={"nom": "club"})

buteurs = (
    joueurs_c[joueurs_c["saison"] == "2025-2026"]
    .merge(noms_clubs, on="equipe_id", how="left")
    .sort_values(["buts", "passes_decisives"], ascending=False)
    .head(5)
    [["prenom", "nom", "club", "buts", "passes_decisives"]]
)
buteurs

,prenom,nom,club,buts,passes_decisives
266,Modou,Thiam,Niayes Académie,19,2
27,Bakary,Wade,Étoile du Cap-Vert,17,5
154,Bamba,Badji,Océan Club,15,5
552,Amadou,Kâ,Banlieue Espoir,15,1
124,Omar,Cissé,Saloum United,15,0


**Interprétation :** Modou Thiam (Niayes Académie) est meilleur buteur avec 19 buts, devant Bakary Wade (Étoile du Cap-Vert, 17 buts et 5 passes). Trois joueurs suivent à 15 buts, départagés ici par leurs passes décisives. Aucun des cinq ne joue pour le champion Presqu'île Racing, dont le titre repose donc davantage sur la solidité défensive (26 buts encaissés seulement).

### 4.2 Avantage du terrain

In [24]:
issue = np.select(
    [matchs_c["buts_dom"] > matchs_c["buts_ext"], matchs_c["buts_dom"] == matchs_c["buts_ext"]],
    ["Victoire domicile", "Nul"],
    default="Victoire extérieur"
)
avantage = pd.crosstab(matchs_c["saison"], issue, normalize="index").round(3)
avantage

col_0,Nul,Victoire domicile,Victoire extérieur
saison,,,
2023-2024,0.250,0.482,0.268
2024-2025,0.226,0.484,0.289
2025-2026,0.305,0.432,0.263


**Interprétation :** l'avantage du terrain est net et stable : les équipes qui reçoivent gagnent environ 48 % de leurs matchs en 2023-2024 et 2024-2025, contre seulement 26 à 29 % pour les visiteurs. Il s'atténue en 2025-2026 (43 % de victoires à domicile), saison marquée par une hausse des nuls (30,5 %).

### 4.3 Affluence et taux de remplissage 2025-2026

In [25]:
affluence = (
    matchs_c[matchs_c["saison"] == "2025-2026"]
    .groupby("equipe_dom_id")["spectateurs"].mean()
    .reset_index()
    .rename(columns={"equipe_dom_id": "equipe_id", "spectateurs": "affluence_moyenne"})
    .merge(equipes[["equipe_id", "nom", "stade", "capacite_stade"]], on="equipe_id", how="left")
)
affluence["affluence_moyenne"] = affluence["affluence_moyenne"].round().astype(int)
affluence["taux_remplissage_pct"] = (affluence["affluence_moyenne"] / affluence["capacite_stade"] * 100).round(1)
affluence = affluence.sort_values("taux_remplissage_pct", ascending=False).reset_index(drop=True)
affluence

,equipe_id,affluence_moyenne,nom,stade,capacite_stade,taux_remplissage_pct
0,E01,14559,Étoile du Cap-Vert,Stade du Cap,18000,80.9
1,E20,6213,Banlieue Espoir,Stade de la Banlieue,8000,77.7
2,E09,8305,Presqu'île Racing,Stade de la Corniche,11000,75.5
3,E16,4415,Forêt FC,Stade de la Forêt,6000,73.6
4,E04,7123,Palmiers FC,Stade des Palmiers,10000,71.2
5,E05,5607,Saloum United,Stade du Saloum,8000,70.1
6,E10,2760,Niayes Académie,Stade des Niayes,4000,69.0
7,E02,8276,Baobab Sporting,Stade des Baobabs,12000,69.0
8,E07,4069,Sahel Olympique,Stade du Sahel,6000,67.8
9,E03,6064,Fleuve Athletic,Stade du Fleuve,9000,67.4


**Interprétation :** les trois clubs qui remplissent le mieux leur stade sont l'Étoile du Cap-Vert (80,9 %), Banlieue Espoir (77,7 %) et Presqu'île Racing (75,5 %), trois clubs du haut du classement : les bons résultats semblent attirer le public. Le taux moyen de la Ligue est d'environ 68 %, aucun stade n'est donc rempli en moyenne.

### 4.4 Quand marque-t-on ?

In [26]:
tranches = pd.cut(buts_c["minute_num"],
                  bins=[0, 15, 30, 45, 60, 75, 90],
                  labels=["1-15", "16-30", "31-45", "46-60", "61-75", "76-90"])
repartition = tranches.value_counts(normalize=True, sort=False).round(3)
print(repartition)

part_add = (buts_c["temps_add"] > 0).mean()
print(f"\nButs dans le temps additionnel : {(buts_c['temps_add'] > 0).sum()} soit {part_add:.1%}")

# Nombre de buts par tranche, en séparant temps réglementaire et temps additionnel
pd.crosstab(tranches, buts_c["temps_add"] > 0, margins=True).rename(
    columns={False: "temps_reglementaire", True: "temps_additionnel", "All": "total"})

minute_num
1-15     0.150
16-30    0.148
31-45    0.185
46-60    0.133
61-75    0.161
76-90    0.225
Name: proportion, dtype: float64

Buts dans le temps additionnel : 227 soit 8.6%


temps_add,temps_reglementaire,temps_additionnel,total
minute_num,,,
1-15,397,0,397
16-30,392,0,392
31-45,448,43,491
46-60,352,0,352
61-75,426,0,426
76-90,412,184,596
All,2427,227,2654


**Interprétation :** la tranche 76-90 semble la plus prolifique (22,5 % des buts), mais elle inclut les 184 buts marqués après la 90e minute (« 90+x ») ; sans eux, elle retombe à un niveau comparable aux autres tranches. De même, la tranche 31-45 contient 43 buts du type « 45+x ». Au total, 227 buts (8,6 %) sont marqués dans le temps additionnel, alors qu'il ne dure que quelques minutes par mi-temps : c'est une période très décisive, qui explique à elle seule l'avance des fins de mi-temps. Découper par `minute_num` regroupe ces buts avec la dernière tranche, ce qu'il faut garder en tête en lisant le tableau.

### 4.5 Penalties et passes

In [27]:
print(f"Part des buts sur penalty : {(buts_c['type_but'] == 'penalty').mean():.1%}")
print(f"Part des buts sans passeur : {buts_c['passeur_id'].isna().mean():.1%}")

# Table d'identité : une ligne par joueur (le nom ne change pas d'une saison à l'autre)
identite = joueurs_c.drop_duplicates("joueur_id")[["joueur_id", "prenom", "nom"]]

duos = (
    buts_c.groupby(["passeur_id", "buteur_id"]).size()
    .nlargest(3, keep="all")
    .rename("nb_buts").reset_index()
    .merge(identite.rename(columns={"joueur_id": "passeur_id", "prenom": "prenom_passeur", "nom": "nom_passeur"}),
           on="passeur_id", how="left")
    .merge(identite.rename(columns={"joueur_id": "buteur_id", "prenom": "prenom_buteur", "nom": "nom_buteur"}),
           on="buteur_id", how="left")
)
duos

Part des buts sur penalty : 10.6%
Part des buts sans passeur : 35.2%


,passeur_id,buteur_id,nb_buts,prenom_passeur,nom_passeur,prenom_buteur,nom_buteur
0,J0441,J0446,6,Malick,Diatta,Khadim,Diop
1,J0240,J0588,5,Thierno,Wade,Bakary,Mendy
2,J0497,J0280,5,Mouhamed,Coly,Modou,Thiam
3,J0553,J0634,5,Youssou,Diallo,Amadou,Kâ
4,J0622,J0280,5,Ousmane,Faye,Modou,Thiam


**Interprétation :** 10,6 % des buts sont marqués sur penalty et 35,2 % n'ont pas de passeur (penalties, actions individuelles, coups francs directs). Le duo le plus prolifique est Malick Diatta → Khadim Diop (6 buts) ; quatre duos suivent à égalité avec 5 buts, ce qui empêche de désigner un 2e et un 3e uniques. Modou Thiam, meilleur buteur 2025-2026, apparaît dans deux de ces duos : il est bien servi par plusieurs coéquipiers.

### 4.6 Masse salariale et résultats (2025-2026)

In [28]:
masse = (
    joueurs_c[joueurs_c["saison"] == "2025-2026"]
    .groupby("equipe_id")["salaire_mensuel_fcfa"].sum()
    .rename("masse_salariale_mensuelle").reset_index()
)
argent = classement_2026.merge(masse, on="equipe_id", how="left")[["rang", "nom", "Pts", "masse_salariale_mensuelle"]]
display(argent.sort_values("masse_salariale_mensuelle", ascending=False))

correlation = argent["masse_salariale_mensuelle"].corr(argent["Pts"])
print(f"Corrélation masse salariale / points : {correlation:.2f}")

,rang,nom,Pts,masse_salariale_mensuelle
1,2,Étoile du Cap-Vert,73,18970000.0
0,1,Presqu'île Racing,75,16830000.0
2,3,Baobab Sporting,67,16010000.0
10,11,Fleuve Athletic,48,14700000.0
3,4,Banlieue Espoir,66,14580000.0
4,5,Palmiers FC,64,14430000.0
9,10,Océan Club,50,13310000.0
13,14,Diourbel Sporting,45,12520000.0
7,8,Saloum United,53,12040000.0
5,6,Niayes Académie,59,11750000.0


Corrélation masse salariale / points : 0.82


**Interprétation :** la corrélation entre masse salariale et points est forte et positive (0,82) : les trois clubs qui paient le plus (Étoile du Cap-Vert, Presqu'île Racing, Baobab Sporting) occupent exactement le podium. On peut conclure qu'en 2025-2026, masse salariale élevée et bons résultats vont ensemble. On ne peut pas conclure que l'argent *cause* les victoires : la relation peut aller dans l'autre sens (un club qui gagne attire des revenus et augmente ses salaires) ou venir d'un troisième facteur (taille du club, nombre de supporters). De plus, avec 20 clubs et une seule saison, l'échantillon est petit, et des exceptions existent (Fleuve Athletic, 4e masse salariale, n'est que 11e).

### 4.7 Carrières

In [29]:
carrieres = joueurs_c.groupby("joueur_id").agg(nb_saisons=("saison", "nunique"),
                                               nb_clubs=("equipe_id", "nunique"))
print("Joueurs différents sur 3 saisons :", len(carrieres))
print("Joueurs ayant joué les 3 saisons  :", (carrieres["nb_saisons"] == 3).sum())
print("Joueurs passés par plusieurs clubs:", (carrieres["nb_clubs"] > 1).sum())

Joueurs différents sur 3 saisons : 717
Joueurs ayant joué les 3 saisons  : 417
Joueurs passés par plusieurs clubs: 69


**Interprétation :** la Ligue a compté 717 joueurs différents en trois saisons, pour 560 places par saison. 417 joueurs (58 %) ont joué les trois saisons, ce qui montre une forte stabilité des effectifs, et seuls 69 joueurs ont porté le maillot de plusieurs clubs : les transferts internes à la Ligue restent rares.

### 4.8 Contrôles croisés

In [30]:
# Score de chaque équipe dans chaque match (une ligne par match et par équipe)
scores = pd.concat([
    matchs_c[["match_id", "equipe_dom_id", "buts_dom"]].set_axis(["match_id", "equipe_id", "score"], axis=1),
    matchs_c[["match_id", "equipe_ext_id", "buts_ext"]].set_axis(["match_id", "equipe_id", "score"], axis=1),
], ignore_index=True)

def controle_scores(table_buts):
    """Renvoie les (match, équipe) dont le nombre de lignes de buts diffère du score."""
    compte = table_buts.groupby(["match_id", "equipe_id"]).size().rename("nb_lignes_buts").reset_index()
    comparaison = scores.merge(compte, on=["match_id", "equipe_id"], how="left")
    comparaison["nb_lignes_buts"] = comparaison["nb_lignes_buts"].fillna(0).astype(int)
    return comparaison[comparaison["score"] != comparaison["nb_lignes_buts"]]

# Contrôle 1 : scores de matchs vs lignes de buts (données nettoyées)
ecarts_propres = controle_scores(buts_c)
print("Écarts score / buts (nettoyé) :", len(ecarts_propres))

# Contrôle 2 : colonne buts de joueurs vs buts comptés dans la table buts
buts_par_joueur = (buts_c.groupby(["saison", "buteur_id"]).size()
                   .rename("buts_fichier").reset_index()
                   .rename(columns={"buteur_id": "joueur_id"}))
comp_joueurs = joueurs_c[["saison", "joueur_id", "buts"]].merge(buts_par_joueur, on=["saison", "joueur_id"], how="left")
comp_joueurs["buts_fichier"] = comp_joueurs["buts_fichier"].fillna(0).astype(int)
print("Écarts joueurs / buts (nettoyé):", (comp_joueurs["buts"] != comp_joueurs["buts_fichier"]).sum())

# Contrôle 1 refait avec le fichier brut
ecarts_brut = controle_scores(buts)
print("Écarts score / buts (brut)     :", len(ecarts_brut))
ecarts_brut

Écarts score / buts (nettoyé) : 0
Écarts joueurs / buts (nettoyé): 0
Écarts score / buts (brut)     : 6


,match_id,equipe_id,score,nb_lignes_buts
31,M0032,E16,3,4
643,M0644,E16,1,2
743,M0744,E06,5,6
777,M0778,E12,1,2
1345,M0206,E14,1,2
1564,M0425,E12,1,2


**Interprétation :** sur les données nettoyées, les deux contrôles donnent 0 écart : chaque score de `matchs` correspond exactement au nombre de buts enregistrés, et la colonne `buts` de `joueurs` est cohérente avec la table `buts`. Avec le fichier `buts.csv` brut, 6 scores ne correspondent plus : chacun a une ligne de but en trop. Ce sont les 6 buts dupliqués supprimés en 2.1, qui gonflaient artificiellement le score de 6 équipes. Ce contrôle confirme que la suppression des doublons était nécessaire et suffisante.

## Partie 5 — Export

In [31]:
with pd.ExcelWriter("rapport_ligue.xlsx") as writer:
    classements.to_excel(writer, sheet_name="Classements", index=False)
    buteurs.to_excel(writer, sheet_name="Buteurs_2025-2026", index=False)
    affluence.to_excel(writer, sheet_name="Affluence_2025-2026", index=False)

# Vérification : on relit le fichier et on affiche la taille de chaque feuille
verif = pd.read_excel("rapport_ligue.xlsx", sheet_name=None)
for feuille, contenu in verif.items():
    print(f"{feuille:22s} : {contenu.shape[0]} lignes × {contenu.shape[1]} colonnes")

Classements            : 60 lignes × 12 colonnes
Buteurs_2025-2026      : 5 lignes × 5 colonnes
Affluence_2025-2026    : 20 lignes × 6 colonnes


**Interprétation :** le fichier `rapport_ligue.xlsx` contient les trois feuilles demandées : les classements des trois saisons (60 lignes, 20 clubs × 3 saisons), les 5 meilleurs buteurs 2025-2026 et l'affluence des 20 clubs en 2025-2026 avec leur taux de remplissage. La relecture du fichier confirme que l'export est complet.

## Bonus — La course au titre 2025-2026

### B.1 Points cumulés journée par journée

In [32]:
course_detail = (resultats[resultats["saison"] == "2025-2026"]
                 .sort_values(["journee", "equipe_id"]).copy())

par_club = course_detail.groupby("equipe_id")
course_detail["pts_cumules"]  = par_club["points"].cumsum()
course_detail["diff_cumulee"] = par_club["buts_pour"].cumsum() - par_club["buts_contre"].cumsum()
course_detail["bp_cumules"]   = par_club["buts_pour"].cumsum()
course_detail = course_detail.merge(equipes[["equipe_id", "nom"]], on="equipe_id", how="left")

course_detail[course_detail["nom"] == "Presqu'île Racing"][["journee", "points", "pts_cumules"]].tail()

,journee,points,pts_cumules
668,34,0,66
688,35,0,66
708,36,3,69
728,37,3,72
748,38,3,75


**Interprétation :** chaque ligne donne le total de points d'un club après une journée donnée ; à la 38e journée, on retrouve bien les 75 points de Presqu'île Racing du classement final, ce qui valide le calcul.

### B.2 Tableau de la course et leader après chaque journée

In [33]:
# Tableau : une ligne par journée, une colonne par club
course = course_detail.pivot_table(index="journee", columns="nom", values="pts_cumules")
display(course.head())

# Leader après chaque journée, avec les critères du règlement pour départager les égalités
leaders = (course_detail
           .sort_values(["journee", "pts_cumules", "diff_cumulee", "bp_cumules"],
                        ascending=[True, False, False, False])
           .groupby("journee").first()[["nom", "pts_cumules", "diff_cumulee"]])
display(leaders)

changements = (leaders["nom"] != leaders["nom"].shift()).sum() - 1
print("Nombre de changements de leader :", changements)

nom,Banlieue Espoir,Baobab Sporting,Delta Club,Diourbel Sporting,Falaise AC,Ferlo United,Fleuve Athletic,Forêt FC,Kaffrine Athletic,Lagune Sportive,Niayes Académie,Océan Club,Palmiers FC,Presqu'île Racing,Sahel Olympique,Saloum United,Savane FC,Sine Racing,Sédhiou Olympique,Étoile du Cap-Vert
journee,,,,,,,,,,,,,,,,,,,,
1,0.0,1.0,0.0,0.0,0.0,3.0,3.0,1.0,0.0,1.0,1.0,3.0,3.0,3.0,0.0,1.0,1.0,1.0,1.0,3.0
2,1.0,1.0,3.0,3.0,1.0,4.0,4.0,2.0,1.0,4.0,4.0,3.0,3.0,3.0,1.0,2.0,1.0,4.0,1.0,6.0
3,4.0,4.0,3.0,3.0,1.0,7.0,4.0,2.0,1.0,5.0,4.0,4.0,3.0,6.0,4.0,5.0,1.0,7.0,4.0,9.0
4,7.0,4.0,4.0,3.0,4.0,7.0,4.0,3.0,2.0,8.0,7.0,4.0,3.0,9.0,7.0,5.0,1.0,10.0,5.0,12.0
5,10.0,7.0,7.0,3.0,4.0,8.0,7.0,4.0,3.0,8.0,7.0,4.0,3.0,12.0,10.0,6.0,2.0,13.0,5.0,13.0


,nom,pts_cumules,diff_cumulee
journee,,,
1,Presqu'île Racing,3,4
2,Étoile du Cap-Vert,6,7
3,Étoile du Cap-Vert,9,9
4,Étoile du Cap-Vert,12,10
5,Étoile du Cap-Vert,13,10
6,Sine Racing,16,7
7,Sine Racing,19,8
8,Presqu'île Racing,19,10
9,Étoile du Cap-Vert,20,12


Nombre de changements de leader : 7


**Interprétation :** la première place a changé 7 fois de mains au cours de la saison. Presqu'île Racing a mené la plus longue série (journées 13 à 30), mais l'Étoile du Cap-Vert lui a repris la tête de la 31e à la 36e journée. Presqu'île Racing ne s'est assuré le titre que lors des deux dernières journées : la course a été indécise jusqu'au bout.

### B.3 Conservation du tableau


Le DataFrame `course` (38 journées × 20 clubs) est conservé pour tracer la course au titre lors de la séance 3.